# Data Processing And Forecasting

This notebook is now a thin orchestration layer around the reusable pipeline code in `src/`. Use it for interactive runs, but keep core logic in the modules so tests and RL use the same implementation.


In [ ]:
from pathlib import Path

import pandas as pd

from src.data_pipeline import (
    FeatureConfig,
    LatentDemandConfig,
    build_hourly_features,
    reconstruct_latent_demand,
    split_and_scale_features,
)
from src.forecasting import (
    ForecastConfig,
    PersistenceForecaster,
    choose_best_forecaster,
    export_rl_forecast_features,
    prepare_forecast_frames,
)


ROOT = Path.cwd()


## 1. Load Data

Point `raw_df` at a dataframe with the FreshRetailNet daily schema used in the original notebook.


In [ ]:
# Example:
# splits = {'train': 'data/train.parquet', 'eval': 'data/eval.parquet'}
# train_df = pd.read_parquet("hf://datasets/Dingdong-Inc/FreshRetailNet-50K/" + splits["train"])
# eval_df = pd.read_parquet("hf://datasets/Dingdong-Inc/FreshRetailNet-50K/" + splits["eval"])
# raw_df = pd.concat([train_df, eval_df], ignore_index=True)

raw_df = pd.DataFrame()
raw_df.head()


## 2. Reconstruct Latent Demand


In [ ]:
latent_config = LatentDemandConfig()
reconstructed_df, reconstruction_diagnostics = reconstruct_latent_demand(raw_df, latent_config)
reconstruction_diagnostics


## 3. Build Hourly Features And Splits


In [ ]:
feature_config = FeatureConfig()
hourly_df = build_hourly_features(reconstructed_df, feature_config)
hourly_df, scaler = split_and_scale_features(hourly_df, feature_config)
hourly_df.head()


## 4. Prepare Forecast Data And Evaluate Candidate Models


In [ ]:
forecast_config = ForecastConfig()
bundle = prepare_forecast_frames(hourly_df, forecast_config)

# Replace or extend with NeuralForecast adapters for LSTM/TFT when running full training.
metrics, val_predictions, best_adapter = choose_best_forecaster(
    [PersistenceForecaster()],
    bundle,
    use_log_target=forecast_config.use_log_target,
)
metrics


## 5. Export Artifacts For RL


In [ ]:
artifacts_dir = ROOT / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)

rl_forecasts = export_rl_forecast_features(val_predictions)
hourly_df.to_parquet(artifacts_dir / 'hourly_features.parquet', index=False)
rl_forecasts.to_parquet(artifacts_dir / 'rl_forecast_features.parquet', index=False)

print('Saved:', artifacts_dir / 'hourly_features.parquet')
print('Saved:', artifacts_dir / 'rl_forecast_features.parquet')
